## Part 2: Data-engineering questions (P2)

Three questions answered with multi-table joins. Each one ends with the answer, the plots, and short notes on
**grain** (what one row is), **missingness** (what is empty and why) and the **pre-race cutoff** (what is known before the race starts).

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED
# Standalone fallback; the automatic merge drops this LOCAL SETUP cell,
# so the merged notebook uses P1's single authoritative flag.
if "EXCLUDE_SHARED_DRIVES" not in globals():
    EXCLUDE_SHARED_DRIVES = False


In [ ]:
import pandas as pd               # tables
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # heatmap (backup section)

clean = load_clean(exclude_shared_drives=EXCLUDE_SHARED_DRIVES)
base = clean["model_base"]    # pre-race columns + the target `podium`
outcomes = clean["outcomes"]  # post-race columns (status, points...). Never features
tables = clean["tables"]      # all other cleaned tables (races, circuits, drivers...)

### Q1: Which circuits convert a front-row start into a podium most often?

**In plain words:** when a driver starts 1st or 2nd (the front row), how often do they finish in the top 3, at each circuit?
Does that ranking change after 2014? We answer it twice: once with **grid** (where the car really started) and once with
**qualifying position** (where it qualified).

**Why 2014?** In 2014 F1 switched to **hybrid engines** (a small turbo engine plus batteries and electric motors).
One team (Mercedes) became much faster than the rest, and every engine part became limited, which caused many more grid penalties.

**Which years count as "before 2014"?** We make three comparisons, each one for grid and for qualifying (6 in total):

| Before | After | Why |
|---|---|---|
| **2004-2013** | 2014-2024 | **Main, the fairest:** similar length (10 vs 11 seasons), grid penalties exist in both, full qualifying data, similar car reliability |
| 1994-2013 | 2014-2024 | from the first season with qualifying data |
| 1950-2013 | 2014-2024 | all of F1 history |

**Tables and joins (multi-hop):**

```
results ──raceId──► races ──circuitId──► circuits        (which circuit was the race at?)
   │
   └──(raceId, driverId)──► qualifying                     (where did the driver qualify?)
```

- `results ⋈ races ⋈ qualifying` is already done inside `load_clean()` → `model_base`
  (`grid`, `podium`, `year`, `circuitId`, `qual_position`). `load_clean()` also used `status` to remove drivers who did not start,
  removed the Indianapolis 500 and kept one row per driver per race.
- Here we add the last hop: `model_base ⋈ circuits` on `circuitId` (circuit name and country).
- `drivers` and `races` are joined only to show names in an example table; `outcomes` only to explain the result (never as a feature).
- Not used: `sprint_results` (a different, shorter race with its own grid), standings, lap times, pit stops.

In [ ]:
# 1) take only the 3 columns we need from the circuits table
circuits = tables["circuits"][["circuitId", "name", "country"]]
# 2) rename "name" to "circuit" so the column name is clear
circuits = circuits.rename(columns={"name": "circuit"})

# 3) join: every row of base gets the name and country of its circuit
q1 = base.merge(circuits, on="circuitId", how="left", validate="many_to_one")

# 4) safety checks: the join must not add or lose rows, and every race must find its circuit
assert len(q1) == len(base), "the join changed the number of rows"
assert q1["circuit"].notna().all(), "a race has no circuit name"

print("rows (one per driver per race):", len(q1))
print("seasons:", q1["year"].min(), "-", q1["year"].max(), "| races:", q1["raceId"].nunique(), "| circuits:", q1["circuit"].nunique())

#### Grid vs qualifying position

- **Qualifying position** = the result of Saturday's timed session (fastest lap = 1st).
- **Grid** = where the car really started on Sunday.
- **Before 2004 they were the same:** the grid was simply the qualifying order.
- **From 2004** the rules limit engines (and from 2008 gearboxes). A driver who changes one early is moved back on the grid
  (a **penalty**), and the drivers behind move up. A car changed after qualifying starts from the **pit lane** (`grid = 0`).
- **After 2014** every part of the hybrid engine is limited, so penalties became about **twice as common**.

The chart shows this per season (from 1994, the first season with qualifying data, so we can see the "before").

In [ ]:
# keep only rows that have a qualifying position (1994 and later)
with_quali = q1[q1["has_qualifying"]].copy()

# places lost between qualifying and the start (positive = started further back)
with_quali["places_lost"] = with_quali["grid"] - with_quali["qual_position"]

# pushed back = lost 3 or more places, or started from the pit lane (grid 0)
with_quali["pushed_back"] = (with_quali["places_lost"] >= 3) | (with_quali["grid"] == 0)

# per season: number of races, number of pushed-back drivers, and drivers per race
per_year = with_quali.groupby("year").agg(races=("raceId", "nunique"), pushed_back=("pushed_back", "sum"))
per_year["per_race"] = per_year["pushed_back"] / per_year["races"]

plt.figure(figsize=(11, 4))
plt.bar(per_year.index, per_year["per_race"], color="tab:red", label="drivers pushed back per race")
plt.axvline(2003.5, color="grey", linestyle="--", label="2004: first engine rule")
plt.axvline(2013.5, color="black", linestyle="--", label="2014: hybrid engines")
plt.title("Drivers who started 3+ places behind their qualifying position (or from the pit lane)")
plt.xlabel("season")
plt.ylabel("drivers per race")
plt.legend()
plt.show()

**Why 3 places and not 1?** The table counts, per race, the drivers who started 1, 2, 3-4 or 5+ places behind their
qualifying position. Losing **1** place happens about equally often in every span, even before 2004 when there were no
penalties, so it is not a penalty (small recording differences, or someone moved ahead after qualifying). Losing **3 or more**
is zero before 2004 and then grows with the penalty rules. 3 places is also the smallest common real penalty
(for example, blocking another driver in qualifying).

In [ ]:
# three time spans: before penalties, first engine/gearbox rules, hybrid era
with_quali["span"] = "1994-2003"
with_quali.loc[with_quali["year"] >= 2004, "span"] = "2004-13"
with_quali.loc[with_quali["year"] >= 2014, "span"] = "2014-24"

# only drivers who started behind their qualifying position (pit-lane starts have a negative value, so they are left out)
lost = with_quali[with_quali["places_lost"] >= 1].copy()

# put every driver in a group by how many places they lost
lost["places"] = "1"
lost.loc[lost["places_lost"] == 2, "places"] = "2"
lost.loc[lost["places_lost"] >= 3, "places"] = "3-4"
lost.loc[lost["places_lost"] >= 5, "places"] = "5+"

# count the drivers per span and group, then divide each row by the number of races in that span
counts = pd.crosstab(lost["span"], lost["places"])
races_per_span = with_quali.groupby("span")["raceId"].nunique()
places_lost_per_race = counts.div(races_per_span, axis=0)

print("drivers per race who started 1, 2, 3-4 or 5+ places behind their qualifying position:")
places_lost_per_race.round(2)

On the **front row**, every driver who qualified 1st/2nd or started 1st/2nd is one of three cases:
**same** (qualified and started on the front row), **penalty** (qualified on the front row but started further back),
or **moved up** (qualified 3rd or lower but started on the front row because someone ahead was penalised).

In [ ]:
# True / False for every row: qualified 1st or 2nd? started 1st or 2nd?
qualified_front = (with_quali["qual_position"] == 1) | (with_quali["qual_position"] == 2)
started_front = (with_quali["grid"] == 1) | (with_quali["grid"] == 2)

# give every row a readable name for its case
SAME = "qualified 1st/2nd and started 1st/2nd (same)"
PENALTY = "qualified 1st/2nd but started further back (penalty)"
MOVED_UP = "qualified 3rd or lower but started 1st/2nd (moved up)"
with_quali["case"] = "not on the front row"
with_quali.loc[qualified_front & started_front, "case"] = SAME
with_quali.loc[qualified_front & ~started_front, "case"] = PENALTY
with_quali.loc[~qualified_front & started_front, "case"] = MOVED_UP

# only drivers on the front row in at least one of the two, counted per span
front_cases = with_quali[qualified_front | started_front]
case_table = pd.crosstab(front_cases["case"], front_cases["span"])
case_table

In [ ]:
# a few real examples, with driver and race names
drivers = tables["drivers"][["driverId", "forename", "surname"]]
races = tables["races"][["raceId", "name"]].rename(columns={"name": "race"})

examples = front_cases[front_cases["case"] != SAME]
examples = examples.merge(drivers, on="driverId", how="left")
examples = examples.merge(races, on="raceId", how="left")
examples = examples.sort_values("year", ascending=False)
examples[["year", "race", "forename", "qual_position", "grid", "case"]].head(6)

In [ ]:
# missing data: % of drivers with a known qualifying position, per season
coverage = q1.groupby("year")["has_qualifying"].mean() * 100

plt.figure(figsize=(11, 3.5))
plt.bar(coverage.index, coverage.values, color="tab:blue", label="% of drivers with a known qualifying position")
plt.title("Qualifying data per season: none before 1994, patchy 1996-2002, complete from 2003")
plt.xlabel("season")
plt.ylabel("% of drivers")
plt.legend(loc="upper left")
plt.show()

#### Filling the qualifying gaps with the grid (only before 2004)

Qualifying positions are missing before 1994 and for many races in 1996-2002. Qualifying **did** happen in those years and
**decided the grid**; the dataset just does not record it. Before 2004 there were no grid penalties, so the grid is the
qualifying order. The check below proves it on the years where we have both. So we make a new column `qual_or_grid`:
the qualifying position, and where it is missing **before 2004**, the grid. Missing values from 2004 on are **not** filled,
because from then on the grid can differ from qualifying (penalties).

In [ ]:
# check: in 1994-2003 (before penalties), is the grid the same as the qualifying position?
before_penalties = q1[q1["has_qualifying"] & (q1["year"] <= 2003)]
same_position = before_penalties["grid"] == before_penalties["qual_position"]
print("1994-2003: grid = qualifying position for", round(same_position.mean() * 100, 1), "% of drivers")

qualified_front_old = (before_penalties["qual_position"] == 1) | (before_penalties["qual_position"] == 2)
started_front_old = (before_penalties["grid"] == 1) | (before_penalties["grid"] == 2)
print("front row: qualified 1st/2nd:", qualified_front_old.sum(),
      "| of these, started 1st/2nd:", (qualified_front_old & started_front_old).sum())

# new column: the qualifying position; where it is missing before 2004, use the grid instead
q1["qual_or_grid"] = q1["qual_position"]
fill = q1["qual_or_grid"].isna() & (q1["year"] < 2004)
q1.loc[fill, "qual_or_grid"] = q1.loc[fill, "grid"]

print("rows filled with the grid:", fill.sum())
print("still missing (2004 and later, not filled):", q1["qual_or_grid"].isna().sum())

#### Helper functions (our toolbox)

| Function | What it does |
|---|---|
| `add_era(df, start_year)` | keeps the seasons from `start_year` on and labels them "start_year-2013" or "2014-24" |
| `front_row_rate(df, position_column)` | per circuit and era: **starts** (drivers in position 1 or 2), **podiums**, **rate** = podiums ÷ starts, **enough_starts** (at least `MIN_STARTS`) |
| `overall_rate(df, position_column)` | the same, but all circuits together |
| `plot_top10(ax, table, era, title)` | bar chart of the 10 circuits with the highest rate in one era |
| `compare_eras(table, before)` | circuits with enough starts in **both** eras, and the **Spearman** correlation of their two rankings |
| `slope_chart(ax, rates, spearman, title)` | the rank-change chart: one line per circuit from its rank before to its rank after |

**Spearman** compares two rankings: **1** = exactly the same order, **0** = the old order says nothing about the new order,
**−1** = the order is reversed.

In [ ]:
MIN_STARTS = 10   # a circuit needs at least 10 front-row starts (about 5 races) in an era to be ranked


def add_era(df, start_year):
    """Keep the seasons from start_year on and label each row with its era."""
    part = df[df["year"] >= start_year].copy()
    part["era"] = str(start_year) + "-2013"
    part.loc[part["year"] >= 2014, "era"] = "2014-24"
    return part


def front_row_rate(df, position_column):
    """For every circuit and era: front-row starts, podiums and the rate."""
    # keep only drivers in position 1 or 2 (a missing position is not front row)
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    # count the starts and add up the podiums (podium is 1 or 0) per circuit and era
    table = front.groupby(["circuit", "era"])["podium"].agg(starts="count", podiums="sum")
    table = table.reset_index()
    # rate = podiums / starts
    table["rate"] = table["podiums"] / table["starts"]
    # enough starts to be ranked?
    table["enough_starts"] = table["starts"] >= MIN_STARTS
    return table


def overall_rate(df, position_column):
    """Front-row starts, podiums and rate per era, all circuits together."""
    is_front = ((df[position_column] == 1) | (df[position_column] == 2)).fillna(False)
    front = df[is_front]
    table = front.groupby("era")["podium"].agg(starts="count", podiums="sum")
    table["rate"] = table["podiums"] / table["starts"]
    return table


def plot_top10(ax, table, era, title):
    """Bar chart of the 10 circuits with the highest rate in one era (only circuits with enough starts)."""
    rows = table[(table["era"] == era) & table["enough_starts"]]
    # sort from low to high and keep the last 10, so the best circuit is drawn at the top
    top10 = rows.sort_values(["rate", "starts"]).tail(10)
    ax.barh(top10["circuit"], top10["rate"] * 100, color="tab:green", label="% of front-row starters on the podium")
    # write "podiums of starts" on each bar
    for i in range(len(top10)):
        ax.text(1, i, str(top10["podiums"].iloc[i]) + " of " + str(top10["starts"].iloc[i]),
                va="center", color="white", fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_title(title)
    ax.set_xlabel("% of front-row starters who finished on the podium")
    ax.legend(loc="lower right", fontsize=8)


def compare_eras(table, before, after="2014-24", min_starts=MIN_STARTS):
    """Rates of the circuits that have enough starts in both eras, and the Spearman correlation of their ranks."""
    rates = table.pivot(index="circuit", columns="era", values="rate")[[before, after]]
    starts = table.pivot(index="circuit", columns="era", values="starts")[[before, after]]
    in_both = (starts[before] >= min_starts) & (starts[after] >= min_starts)
    rates = rates[in_both]
    ranks = rates.rank(ascending=False)   # 1 = highest rate
    spearman = ranks[before].corr(ranks[after])
    return rates, spearman


def slope_chart(ax, rates, spearman, title):
    """Left column = order before, right column = order after, one line per circuit."""
    before = rates.columns[0]
    after = rates.columns[1]
    # position of every circuit in each column (0 = top)
    left = pd.Series(range(len(rates)), index=rates.sort_values(before, ascending=False).index)
    right = pd.Series(range(len(rates)), index=rates.sort_values(after, ascending=False).index)
    for circuit in rates.index:
        if right[circuit] < left[circuit]:
            color = "tab:green"
        elif right[circuit] > left[circuit]:
            color = "tab:red"
        else:
            color = "grey"
        ax.plot([0, 1], [left[circuit], right[circuit]], marker="o", color=color)
        ax.text(-0.03, left[circuit], circuit + " (" + str(round(rates.loc[circuit, before] * 100)) + "%)",
                ha="right", va="center", fontsize=8)
        ax.text(1.03, right[circuit], "(" + str(round(rates.loc[circuit, after] * 100)) + "%) " + circuit,
                ha="left", va="center", fontsize=8)
    ax.plot([], [], color="tab:green", label="better rank after")
    ax.plot([], [], color="tab:red", label="worse rank after")
    ax.plot([], [], color="grey", label="same rank")
    ax.invert_yaxis()                       # best at the top
    ax.set_xlim(-1.2, 2.2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([before, after])
    ax.set_yticks([])
    ax.set_ylabel("rank (best at the top)")
    ax.set_title(title + "\n" + str(len(rates)) + " circuits, Spearman = " + str(round(spearman, 2)))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=8)

#### Minimum number of starts

Some circuits hosted only a few races in an era. A circuit with 2 front-row starts and 2 podiums shows 100%, but that is
2 cars, not a pattern, and it would wrongly top the ranking. So a circuit is ranked only if it has **at least 10 front-row
starts in that era (about 5 races)**: then one car can change the rate by at most 10 points. A later cell checks that
choosing 6, 8 or 15 instead does not change the answer.

In [ ]:
all_seasons = add_era(q1, 1950)
all_grid_table = front_row_rate(all_seasons, "grid")

# plt.figure(figsize=(8, 4))
# plt.hist(all_grid_table["starts"], bins=30, color="tab:blue", edgecolor="white", label="circuit-eras")
# plt.axvline(MIN_STARTS, color="red", linestyle="--", label="minimum = 10 starts")
# plt.title("Front-row starts per circuit and era (grid 1st/2nd, 1950-2013 and 2014-24)")
# plt.xlabel("front-row starts")
# plt.ylabel("number of circuit-eras")
# plt.legend()
# plt.show()

small = all_grid_table[~all_grid_table["enough_starts"]]
print("circuit-eras with fewer than", MIN_STARTS, "starts:", len(small), "of", len(all_grid_table))
print("examples (these are left out of the ranking):")
small.sort_values("rate", ascending=False).head(5)

#### Main comparison: 2004-2013 vs 2014-2024

The fairest comparison: grid penalties exist in both eras (so grid and qualifying mean the same thing on both sides),
the eras have a similar length, and qualifying data is complete. Top row = grid, bottom row = qualifying.

In [ ]:
fair = add_era(q1, 2004)
grid_fair = front_row_rate(fair, "grid")
qual_fair = front_row_rate(fair, "qual_or_grid")

print("Front row -> podium, all circuits together:")
print("grid:")
display(overall_rate(fair, "grid").loc[["2004-2013", "2014-24"]].round(3))
print("qualifying:")
display(overall_rate(fair, "qual_or_grid").loc[["2004-2013", "2014-24"]].round(3))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
plot_top10(axes[0, 0], grid_fair, "2004-2013", "Top 10 circuits, started 1st/2nd (grid), 2004-2013")
plot_top10(axes[0, 1], grid_fair, "2014-24", "Top 10 circuits, started 1st/2nd (grid), 2014-2024")
plot_top10(axes[1, 0], qual_fair, "2004-2013", "Top 10 circuits, qualified 1st/2nd, 2004-2013")
plot_top10(axes[1, 1], qual_fair, "2014-24", "Top 10 circuits, qualified 1st/2nd, 2014-2024")
plt.tight_layout()
plt.show()

**Does the ranking change?** We take the circuits with at least 10 front-row starts in **both** eras and rank them in each era.
Each line is one circuit: going up (green) = better rank after 2014, going down (red) = worse.

In [ ]:
grid_rates, grid_spearman = compare_eras(grid_fair, "2004-2013")
qual_rates, qual_spearman = compare_eras(qual_fair, "2004-2013")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
slope_chart(axes[0], grid_rates, grid_spearman, "Rank change 2004-13 -> 2014-24, started 1st/2nd (grid)")
slope_chart(axes[1], qual_rates, qual_spearman, "Rank change 2004-13 -> 2014-24, qualified 1st/2nd")
plt.tight_layout()
plt.show()

#### All six comparisons

The same calculation for the three "before" windows (1950-2013, 1994-2013, 2004-2013), each with grid and with
qualifying (`qual_or_grid`), and each compared with **2014-24**.

In [ ]:
# compute the 6 comparisons: 3 "before" windows x 2 positions
rows = []
for start_year in [1950, 1994, 2004]:
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    for position_column in ["grid", "qual_or_grid"]:
        overall = overall_rate(data, position_column)
        rates, spearman = compare_eras(front_row_rate(data, position_column), before)
        rows.append([before, position_column, overall.loc[before, "rate"] * 100,
                     overall.loc["2014-24", "rate"] * 100, spearman])

six = pd.DataFrame(rows, columns=["before", "position", "rate before", "rate 2014-24", "spearman"])
six["position"] = six["position"].replace("qual_or_grid", "qualifying")   # nicer name for the charts

**1) The podium rate: each "before" window next to 2014-24.** One chart per window. In each chart the grey bar is the
window before 2014 and the green bar is 2014-24, for grid and for qualifying.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, before in zip(axes, ["1950-2013", "1994-2013", "2004-2013"]):
    # the 2 rows of this window (grid and qualifying), with the position as row name
    window = six[six["before"] == before].set_index("position")
    # keep the two rates and name them after their years
    pair = window[["rate before", "rate 2014-24"]]
    pair.columns = [before, "2014-24"]

    pair.plot(kind="bar", ax=ax, color=["tab:grey", "tab:green"])
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.0f", fontsize=9)   # write the value on top of each bar

    # difference between the two grid bars, using the same rounded numbers that are written on the bars
    gain = round(pair.loc["grid", "2014-24"]) - round(pair.loc["grid", before])
    ax.set_title(before + " vs 2014-24\n(grid: +" + str(gain) + " points after 2014)")
    ax.set_xlabel("front row by")
    ax.set_ylabel("% of front-row starters on the podium")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

**2) The circuit ranking: how similar is each "before" window to 2014-24?** Spearman: 1 = same order, 0 = unrelated.

In [ ]:
spearman_table = six.pivot(index="before", columns="position", values="spearman")

ax = spearman_table.plot(kind="bar", color=["tab:blue", "tab:orange"], figsize=(8, 4.5))
ax.axhline(0, color="black", linewidth=1)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", fontsize=9)   # write the value on top of each bar
ax.set_title("How similar is the circuit ranking to 2014-24?")
ax.set_xlabel('"before 2014" window')
ax.set_ylabel("Spearman (1 = same order, 0 = unrelated)")
ax.set_ylim(-0.5, 1)
ax.tick_params(axis="x", rotation=0)
plt.show()

#### Why did the front row convert more often?

Every front-row starter ends in one of three ways: **podium**, **did not finish** (the car broke down, crashed or was
disqualified), or **finished 4th or lower**. We also count how often both front-row cars came from the same team.
`outcomes` is used here only to explain the result, never as a model feature.

In [ ]:
# add positionText from outcomes: a number ("1", "2", ...) if the driver was classified, a letter (R = retired...) if not
why = q1.merge(outcomes[["raceId", "driverId", "positionText"]], on=["raceId", "driverId"], how="left")
why = why[(why["grid"] == 1) | (why["grid"] == 2)].copy()

# three spans of years
why["span"] = "1950-2003"
why.loc[why["year"] >= 2004, "span"] = "2004-13"
why.loc[why["year"] >= 2014, "span"] = "2014-24"

# put every front-row starter into one of three groups
why["result"] = "finished 4th or lower"
why.loc[~why["positionText"].str.isnumeric(), "result"] = "did not finish"
why.loc[why["podium"] == 1, "result"] = "podium"

# % of each group per span (normalize="index" makes every row add up to 100%)
result_share = pd.crosstab(why["span"], why["result"], normalize="index") * 100
result_share = result_share[["podium", "finished 4th or lower", "did not finish"]]
display(result_share.round(1))

result_share.plot(kind="bar", stacked=True, color=["tab:green", "tab:orange", "tab:red"], figsize=(7, 4))
plt.title("What happened to the front-row starters (grid 1st/2nd)?")
plt.xlabel("years")
plt.ylabel("% of front-row starters")
plt.xticks(rotation=0)
plt.legend(title="result", bbox_to_anchor=(1, 1))
plt.show()

# how often were both front-row cars from the same team?
per_race = why.groupby(["raceId", "span"]).agg(cars=("constructorId", "count"), teams=("constructorId", "nunique"))
per_race = per_race.reset_index()
per_race = per_race[per_race["cars"] == 2]
per_race["same_team"] = per_race["teams"] == 1
print("% of races where both front-row cars were from the same team:")
print((per_race.groupby("span")["same_team"].mean() * 100).round(1))

#### Check: does our choice of 10 starts change the answer?

The minimum of 10 starts is our own decision. We repeat the main comparison (2004-13 vs 2014-24) with 6, 8 and 15
to see whether the answer (how similar the rankings are, and which circuits are on top after 2014) stays the same.

In [ ]:
rows = []
for min_starts in [6, 8, 10, 16]:
    for position_column, table in [("grid", grid_fair), ("qual_or_grid", qual_fair)]:
        rates, spearman = compare_eras(table, "2004-2013", min_starts=min_starts)
        top5 = rates.sort_values("2014-24", ascending=False).head(5).index
        rows.append([position_column, min_starts, len(rates), round(spearman, 2), ", ".join(top5)])

check = pd.DataFrame(rows, columns=["position", "min starts", "circuits", "spearman", "top 5 in 2014-24"])
with pd.option_context("display.max_colwidth", None):
    display(check)

#### Answer to Q1

**1. Best circuits in the hybrid era (2014-24): Monaco and Suzuka.** About 9 in 10 front-row starters finished on the podium
there: Monaco 17 of 19, Suzuka 16 of 18 (89%). They are the **top 2 in all six comparisons**, with grid and with qualifying.
Next come Yas Marina, Barcelona and Spa (82-86%).

**2. Main comparison, 2004-13 vs 2014-24 (the fairest).** The front row converts more often after 2014:

| Front row by | 2004-13 | 2014-24 |
|---|---|---|
| grid (started 1st/2nd) | 69% (254 of 366) | **76%** (345 of 453) |
| qualifying (qualified 1st/2nd) | 69% (251 of 366) | **74%** (336 of 454) |

The ranking is **only partly the same**. Spearman is **0.28 (grid)** and **0.45 (qualifying)**, for 14 circuits.
- **Moved up:** Yas Marina (60% → 86%), Barcelona, Interlagos, Canada.
- **Moved down:** Monza, Silverstone, Bahrain, Hungaroring.
- Monza and Silverstone barely changed their own rate (75% → 73%, 75% → 75% on grid). They dropped because other circuits rose.
- With a minimum of 6, 8 or 15 starts, Spearman stays positive (0.09-0.39 grid, 0.24-0.54 qualifying), and Monaco and Suzuka
  stay the top 2.

**3. The further back "before" goes, the more different the ranking looks.**

| Before | Spearman grid | Spearman qualifying |
|---|---|---|
| 2004-2013 | 0.28 | 0.45 |
| 1994-2013 | -0.04 | -0.00 |
| 1950-2013 | -0.08 | -0.11 |

Compared with the last decade, part of the order is kept. Compared with all of F1 history, the old order says nothing about the new one.

**4. Why the front row converts more often.**
- Over the long run, front-row cars stopped breaking down: **31%** did not finish in 1950-2003, **10%** in 2004-13, **7%** in 2014-24.
- Between 2004-13 and 2014-24, the share that finished but lost the podium on track also fell (20% → 17%).
- Both front-row cars came from the same team more often (32% → 44% of races). One dominant team kept both spots.

**5. Grid vs qualifying.**
- In 1994-2003 the grid was the qualifying order. 98.5% of drivers started where they qualified, and all 164 front-row
  qualifiers started on the front row.
- Grid penalties start in 2004. Drivers per race who started 3+ places behind their qualifying spot: **0.01** (1994-2003),
  **0.76** (2004-13), **1.05** (2014-24)
  (the 3-4 and 5+ columns of the places-lost table added together).
- On the front row, 0 → 9 → 28 qualifiers were pushed back. Example: Verstappen, Belgium 2024, qualified 1st and started 11th.
- This is why the grid and qualifying answers are identical before 2004 and differ a little after.
- Qualifying gives a slightly lower rate after 2014 (74% vs 76%), because a fast car pushed off the front row still counts
  in qualifying.

**Grain**
- Input: one row per driver per race (`model_base`).
- Front-row subset: 2 rows per race (position 1 and 2).
- Result tables: one row per circuit and era.
- The circuits join is many-to-one, and the row count is checked to stay the same.

**Missingness**
- `qual_position` is empty before 1994 and patchy in 1996-2002 (see the coverage chart).
- `qual_or_grid` fills these gaps with the grid, **only before 2004**, when grid = qualifying order (checked: 164 of 164
  front-row qualifiers started on the front row).
- 21 missing values from 2004 on are not filled.
- `grid` has no gaps. `grid = 0` (pit-lane start) is not counted as front row.

**Pre-race cutoff**
- `grid` and `qual_position` are both known before the race starts.
- `podium` is the outcome we measure, not an input.
- `outcomes` (positionText) is used only to explain why the rate rose, never as an input.

**Limitations**
- Some circuits hosted only a few races in an era (fewer than 10 front-row starts), so they are left out of the ranking.
- One `circuitId` covers all layout changes of a track (e.g. Silverstone, Spa).
- Circuits come and go. Magny-Cours and Istanbul topped 2004-13 but are not on the calendar after 2014.
- Shared drives (two drivers, one car) exist only before 1965 and are counted once, with the best finish.

#### Backup: longer "before" windows and four periods

Extra detail, not needed for the answer above.

In [ ]:
# rank change for the two longer windows: 1994-2013 and 1950-2013 (top = grid, bottom = qualifying)
fig, axes = plt.subplots(2, 2, figsize=(15, 14))
for column, start_year in enumerate([1994, 1950]):
    data = add_era(q1, start_year)
    before = str(start_year) + "-2013"
    rates, spearman = compare_eras(front_row_rate(data, "grid"), before)
    slope_chart(axes[0, column], rates, spearman, "Rank change " + before + " -> 2014-24, grid")
    rates, spearman = compare_eras(front_row_rate(data, "qual_or_grid"), before)
    slope_chart(axes[1, column], rates, spearman, "Rank change " + before + " -> 2014-24, qualifying")
plt.tight_layout()
plt.show()

In [ ]:
# four periods: the rate for all circuits together (grid)
q1["period"] = "1950-93"
q1.loc[q1["year"] >= 1994, "period"] = "1994-2003"
q1.loc[q1["year"] >= 2004, "period"] = "2004-13"
q1.loc[q1["year"] >= 2014, "period"] = "2014-24"

grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
by_period = grid_front.groupby("period")["podium"].agg(starts="count", podiums="sum")
by_period["rate"] = by_period["podiums"] / by_period["starts"]
display(by_period.round(3))

plt.figure(figsize=(7, 4))
plt.bar(by_period.index, by_period["rate"] * 100, color="tab:blue", label="all circuits together")
plt.ylim(0, 100)
plt.title("Front row (grid 1st/2nd) -> podium, per period")
plt.xlabel("period")
plt.ylabel("% of front-row starters on the podium")
plt.legend(loc="upper left")
plt.show()

In [ ]:
# heatmap: rate per circuit and period, only where the circuit had at least 10 front-row starts
grid_front = q1[(q1["grid"] == 1) | (q1["grid"] == 2)]
period_table = grid_front.groupby(["circuit", "period"])["podium"].agg(starts="count", podiums="sum").reset_index()
period_table["rate"] = period_table["podiums"] / period_table["starts"]
period_table = period_table[period_table["starts"] >= MIN_STARTS]
heat = period_table.pivot(index="circuit", columns="period", values="rate") * 100

# keep circuits that appear in at least 2 periods, best in 2014-24 at the top
heat = heat[heat.notna().sum(axis=1) >= 2]
heat = heat.sort_values("2014-24", ascending=False)

plt.figure(figsize=(8, 9))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="Greens", vmin=0, vmax=100,
            cbar_kws={"label": "% of front-row starters on the podium"})
plt.title("Front row (grid 1st/2nd) -> podium per circuit and period\n(blank = fewer than 10 starts)")
plt.xlabel("period")
plt.ylabel("circuit")
plt.show()

### Q2: Do drivers get more podiums at their home circuit?

**In plain words:** does a British driver racing in Britain finish on the podium more often than when racing elsewhere?

**The catch:** if home drivers simply start further forward, they will of course finish better. So we compare
**drivers who started from the same grid row**: a home driver who started 3rd-4th is compared with away drivers who
also started 3rd-4th.

**Tables and joins (multi-hop):**

```
results ──driverId──► drivers                              (nationality of the driver)
   │
   └──raceId──► races ──circuitId──► circuits              (country of the circuit)
```

- `results ⋈ races` is already done inside `load_clean()` → `model_base` (`grid`, `podium`, `year`, `circuitId`).
- Here we add `drivers` (on `driverId`) for the nationality and `circuits` (on `circuitId`) for the country.
- Nationality ("British") and country ("UK") are written differently, so we connect them with a mapping written by hand.
- Not used: qualifying (we control with `grid`, which exists for every race since 1950), standings, lap times, pit stops.
- All seasons, **1950-2024**: nationality, country and grid are never missing.

In [ ]:
# driver nationality (from drivers.csv) -> country name as written in circuits.csv, written by hand
NATIONALITY_TO_COUNTRY = {
    "American": "USA",
    "American-Italian": "USA", 
    "Argentine": "Argentina", 
    "Argentine-Italian": "Argentina",
    "Argentinian": "Argentina", 
    "Australian": "Australia",
    "Austrian": "Austria", 
    "Belgian": "Belgium",
    "Brazilian": "Brazil",
    "British": "UK",
    "Canadian": "Canada", 
    "Chilean": "Chile", 
    "Chinese": "China",
    "Colombian": "Colombia", 
    "Czech": "Czech Republic", 
    "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany",
    "Finnish": "Finland", 
    "French": "France",
    "German": "Germany",
    "Hungarian": "Hungary",
    "Indian": "India",
    "Indonesian": "Indonesia",
    "Irish": "Ireland", 
    "Italian": "Italy", 
    "Japanese": "Japan",
    "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", 
    "Mexican": "Mexico", 
    "Monegasque": "Monaco",
    "New Zealander": "New Zealand", 
    "Polish": "Poland", 
    "Portuguese": "Portugal",
    "Rhodesian": "Rhodesia",
    "Russian": "Russia", 
    "South African": "South Africa",
    "Spanish": "Spain",
    "Swedish": "Sweden",
    "Swiss": "Switzerland",
    "Thai": "Thailand",
    "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}

# drivers with two nationalities have a second home country
SECOND_COUNTRY = {"American-Italian": "Italy", 
                  "Argentine-Italian": "Italy"}

# show the mapping as a table (shared with P3 for the home-race feature)
mapping = pd.DataFrame(list(NATIONALITY_TO_COUNTRY.items()), columns=["nationality", "home_country"])
mapping["second_home_country"] = mapping["nationality"].map(SECOND_COUNTRY)
mapping

In [ ]:
# 1) driver nationality; .str.strip() removes spaces at the ends ("Argentinian " -> "Argentinian")
drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip() #  `strip()` removes spaces at the start and end

# 2) circuit country; "United States" and "USA" are the same country written two ways
circuit_country = tables["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuit"})
circuit_country["country"] = circuit_country["country"].replace("United States", "USA")

# 3) joins: every row of base gets the driver's nationality and the circuit's country
q2 = base.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
q2 = q2.merge(circuit_country, on="circuitId", how="left", validate="many_to_one")

# 4) safety checks
assert len(q2) == len(base), "a join changed the number of rows"
assert q2["nationality"].notna().all(), "a driver has no nationality"
assert q2["country"].notna().all(), "a circuit has no country"
unmapped = set(q2["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, "nationalities missing from the mapping: " + str(unmapped)

# 5) home = the circuit is in the driver's (first or second) home country
q2["home_country"] = q2["nationality"].map(NATIONALITY_TO_COUNTRY) # the mapping is happening here, not in the join, 
q2["second_home_country"] = q2["nationality"].map(SECOND_COUNTRY) # because some drivers have two nationalities and a second home country
q2["is_home"] = (q2["country"] == q2["home_country"]) | (q2["country"] == q2["second_home_country"])

print("rows:", len(q2), "| home rows:", q2["is_home"].sum(), "| away rows:", (~q2["is_home"]).sum())

#### Demo: how the Q2 joins work, step by step

Six real rows, one for each case: a normal home race, a normal away race, two drivers with two nationalities,
a circuit written "United States", and a nationality with a hidden space. Then two fake rows show what would happen
if a driver had no nationality or a circuit had no country. These cells do not change `q2`.

In [ ]:
# DEMO (only to understand the joins): 6 real rows, one for each case. Nothing here changes q2.

# the 6 example rows: (raceId, driverId)
examples = [
    (1132, 1),     # Hamilton (British) at the British GP 2024        -> home
    (1136, 1),     # Hamilton (British) at the Italian GP 2024        -> away
    (754, 491),    # Thiele (American-Italian) at the Italian GP 1960 -> home by his SECOND country
    (776, 573),    # de Tomaso (Argentine-Italian) at the Argentine GP 1957 -> home by his FIRST country
    (1119, 858),   # Sargeant (American) at Las Vegas 2023: circuit written "United States"
    (1136, 861),   # Colapinto ("Argentinian " with a space) at the Italian GP 2024 -> away
]
example_pairs = pd.DataFrame(examples, columns=["raceId", "driverId"])

# STEP 0: the three tables alone (only the rows we need)
demo_base = base.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "circuitId", "year", "grid", "podium"]]
print("STEP 0a: BASE (model_base), our 6 rows")
display(demo_base)

demo_drivers = tables["drivers"][tables["drivers"]["driverId"].isin(demo_base["driverId"])][["driverId", "forename", "surname", "nationality"]].copy()
demo_drivers["nationality in quotes"] = '"' + demo_drivers["nationality"] + '"'   # quotes make the hidden space visible
print("STEP 0b: DRIVERS table, only these drivers (before .str.strip())")
display(demo_drivers)

demo_circuits = tables["circuits"][tables["circuits"]["circuitId"].isin(demo_base["circuitId"])][["circuitId", "name", "country"]]
print("STEP 0c: CIRCUITS table, only these circuits (before the 'United States' -> 'USA' fix)")
display(demo_circuits)

# STEP 1: base + drivers, matched on driverId -> every row gets the nationality (already stripped)
step1 = demo_base.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
print("STEP 1: base + drivers (matched on driverId)")
display(step1)

# STEP 2: + circuits, matched on circuitId -> every row gets the circuit name and its country (already fixed)
step2 = step1.merge(circuit_country, on="circuitId", how="left", validate="many_to_one")
print("STEP 2: + circuits (matched on circuitId)")
display(step2)

# STEP 3: home countries and is_home, exactly like the real cell
step2["home_country"] = step2["nationality"].map(NATIONALITY_TO_COUNTRY)
step2["second_home_country"] = step2["nationality"].map(SECOND_COUNTRY)
step2["is_home"] = (step2["country"] == step2["home_country"]) | (step2["country"] == step2["second_home_country"])
print("STEP 3: home country (first and second) and is_home")
display(step2[["driverId", "nationality", "circuit", "country", "home_country", "second_home_country", "is_home"]])

In [ ]:
# mapping check: which countries never meet?
race_countries = set(q2["country"])
driver_countries = set(q2["home_country"]) | set(q2["second_home_country"].dropna())

print("driver countries that never hosted a race (their drivers are always away):")
print(sorted(driver_countries - race_countries))
print("race countries with no driver from that country (nobody is ever at home there):")
print(sorted(race_countries - driver_countries))
print()
print("home starts per country (top 10):")
q2[q2["is_home"]]["country"].value_counts().head(10)

#### Grid groups

We group the starting positions by **grid row**. On an F1 grid the cars stand in rows of two (1st and 2nd side by side,
then 3rd and 4th, ...), so each group is one row: **1-2, 3-4, 5-6, ... 19-20**. Everything further back, and pit-lane
starts (`grid = 0`), go into the last group **"21+ / pit lane"**. Each group has about 150 home starts, enough to compare.
(The work split suggested 1 / 2-3 / 4-6 / 7-10 / 11+; grid rows give the same answer with a finer view.)

In [ ]:
# the groups in grid order (used to sort tables and charts)
GRID_GROUPS = ["1-2", "3-4", "5-6", "7-8", "9-10", "11-12", "13-14", "15-16", "17-18", "19-20", "21+ / pit lane"]

# every row starts in the last group (grid 21 or further back, or pit lane = grid 0)
q2["grid_group"] = "21+ / pit lane"

# then each pair of grid positions gets its own group
q2.loc[q2["grid"].between(1, 2), "grid_group"] = "1-2"
q2.loc[q2["grid"].between(3, 4), "grid_group"] = "3-4"
q2.loc[q2["grid"].between(5, 6), "grid_group"] = "5-6"
q2.loc[q2["grid"].between(7, 8), "grid_group"] = "7-8"
q2.loc[q2["grid"].between(9, 10), "grid_group"] = "9-10"
q2.loc[q2["grid"].between(11, 12), "grid_group"] = "11-12"
q2.loc[q2["grid"].between(13, 14), "grid_group"] = "13-14"
q2.loc[q2["grid"].between(15, 16), "grid_group"] = "15-16"
q2.loc[q2["grid"].between(17, 18), "grid_group"] = "17-18"
q2.loc[q2["grid"].between(19, 20), "grid_group"] = "19-20"

# count home and away starts in each group
group_counts = pd.crosstab(q2["grid_group"], q2["is_home"])                              # count each (group, home/away) pair
group_counts = group_counts.loc[GRID_GROUPS]                                             # put the groups in grid order
group_counts = group_counts.rename(columns={False: "away starts", True: "home starts"})  # readable column names
group_counts

#### Without grid groups vs with grid groups

First the plain comparison (all home starts vs all away starts). Then the fair comparison inside each grid row.
**"difference"** = home % − away %: how many percentage points higher the home rate is in that row
(for example 45% home and 40% away gives +5; a negative number means away is higher).

In [ ]:
# plain comparison, no grid groups

# split the rows into two groups
home_rows = q2[q2["is_home"]]       # only home races
away_rows = q2[~q2["is_home"]]      # not home races      # only away races (~ means NOT)

# build a small table: one row for home, one row for away
plain = pd.DataFrame({
    "starts": [len(home_rows), len(away_rows)],                          # how many races
    "podiums": [home_rows["podium"].sum(), away_rows["podium"].sum()],   # how many podiums (podium is 1 or 0)
}, index=["home", "away"])

# podium % = podiums / starts * 100
plain["podium %"] = plain["podiums"] / plain["starts"] * 100
plain.round(1)

In [ ]:
# fair comparison: podium rate for home and away inside each grid group
# (home_rows and away_rows come from the plain-comparison cell above)

# count starts and podiums per grid group, separately for home and away
home_away = pd.DataFrame({
    "home starts": home_rows.groupby("grid_group")["podium"].count(),    # how many home races in each group
    "home podiums": home_rows.groupby("grid_group")["podium"].sum(),     # how many of them ended on the podium
    "away starts": away_rows.groupby("grid_group")["podium"].count(),
    "away podiums": away_rows.groupby("grid_group")["podium"].sum(),
})

# put the groups in grid order (1-2, 3-4, ... instead of alphabetical)
home_away = home_away.loc[GRID_GROUPS]

# % = podiums / starts * 100, then difference = home % minus away %
home_away["home %"] = home_away["home podiums"] / home_away["home starts"] * 100
home_away["away %"] = home_away["away podiums"] / home_away["away starts"] * 100
home_away["difference"] = home_away["home %"] - home_away["away %"]
home_away.round(1)

In [ ]:
# grouped bar chart: home (orange) next to away (blue) for each grid row
home_away[["home %", "away %"]].plot(kind="bar", color=["tab:orange", "tab:blue"], figsize=(14, 5))

plt.title("Podium rate at home vs away, for drivers who started from the same grid row (1950-2024)")
plt.xlabel("starting position (grid row)")
plt.ylabel("% of drivers who finished on the podium")
plt.ylim(0, 100)                                       # y axis from 0 to 100%
plt.xticks(rotation=0)                                 # keep the group names horizontal
plt.legend(["home (racing in own country)", "away"])   # legend names, same order as the columns
plt.show()

#### One number for all grid groups: 2 methods

Each method turns the 11 differences into one answer. If both say "about 0", there is no home advantage.

1. **Simple average:** the average of the 11 differences in the table (every grid row counts the same).
2. **Exact grid position:** 1st vs 1st, 2nd vs 2nd ... 20th vs 20th, instead of grid rows.

**How the numbers are calculated, step by step (numbers from the table above):**

**Step 1: the % in each row** = podiums ÷ starts × 100. A sample of three rows:

| grid row | home starts | home podiums | home % | away starts | away podiums | away % |
|---|---|---|---|---|---|---|
| 1-2 | 160 | 96 | 96 ÷ 160 × 100 = **60.0** | 2067 | 1254 | 1254 ÷ 2067 × 100 = **60.7** |
| 3-4 | 144 | 65 | 65 ÷ 144 × 100 = **45.1** | 2080 | 838 | 838 ÷ 2080 × 100 = **40.3** |
| 9-10 | 180 | 8 | 8 ÷ 180 × 100 = **4.4** | 2033 | 145 | 145 ÷ 2033 × 100 = **7.1** |

**Step 2: the difference in each row** = home % − away %:

| grid row | home % − away % | difference | meaning |
|---|---|---|---|
| 1-2 | 60.0 − 60.7 | **−0.7** | away slightly higher |
| 3-4 | 45.1 − 40.3 | **+4.9** | home higher |
| 9-10 | 4.4 − 7.1 | **−2.7** | away higher |

In [ ]:
# METHOD 1: simple average of the 11 differences (every grid row counts the same)
average_difference = home_away["difference"].mean()
print("average difference over the 11 grid rows:", round(average_difference, 2), "percentage points")

In [ ]:
# METHOD 2: exact grid position (1st vs 1st, 2nd vs 2nd, ... 20th vs 20th) instead of grid rows

# only grid positions 1 to 20
home_1_20 = home_rows[home_rows["grid"].between(1, 20)]
away_1_20 = away_rows[away_rows["grid"].between(1, 20)]

# podium % for every grid position, at home and away
by_position = pd.DataFrame({
    "home %": home_1_20.groupby("grid")["podium"].mean() * 100,
    "away %": away_1_20.groupby("grid")["podium"].mean() * 100,
})
by_position["difference"] = by_position["home %"] - by_position["away %"]

print("average difference over grid positions 1-20:", round(by_position["difference"].mean(), 2), "percentage points")
by_position.round(1)

#### Answer to Q2

**No, drivers do not get more podiums at home** once we compare drivers who started from the same grid row.

- **Without grid groups**, home even looks worse: 11.9% podiums at home vs 13.8% away. But home drivers start further back
  more often: 15% of home starts are from 21st or further back / the pit lane, vs 11% of away starts
  (291 of 1932 vs 2542 of 22634, see the grid-group table). Many local drivers entered only their home race.
- **With grid groups** (the fair comparison), home and away are almost the same in every row:
  - front row (1-2): 60.0% home vs 60.7% away
  - 5-6: 21.7% vs 21.9%
  - home is clearly higher in only 2 of 11 rows (3-4: +4.9, 11-12: +1.8)
- **One number, 2 methods** (difference = home % − away %, in percentage points):

  | Method | Result |
  |---|---|
  | 1. Simple average of the 11 grid rows | **−0.24** |
  | 2. Exact grid position (1st vs 1st ... 20th vs 20th) | **−0.20** |

  Both are less than 1 percentage point, so home makes practically no difference.
- The +4.9 in row 3-4 comes from only 144 home starts (65 podiums). It is alone among the other rows, so it is not a home advantage.

**Grain**
- Input: one row per driver per race (`model_base`).
- Result: one row per grid row (11 groups), with home and away side by side.
- Method 2: one row per grid position (1-20).
- Both joins (`drivers`, `circuits`) are many-to-one, and the row count is checked to stay the same.

**Missingness**
- Nationality, circuit country and grid are never missing, so all seasons 1950-2024 are used.
- Every nationality is in the mapping (checked with an assert).
- 14 driver countries never hosted a race (e.g. Finland, New Zealand), so their drivers are always "away".
- 9 race countries never had a driver from there (e.g. Bahrain, Singapore), so nobody is "home" there.
- Method 2 uses grid 1-20 only.

**Pre-race cutoff**
- Nationality, circuit country and grid are all known before the race starts.
- `podium` is the outcome we measure, not an input.
- No column from `outcomes` is used.

**Limitations**
- Nationality is not the same as where a driver lives or grew up.
- Only 2 dual nationalities exist in the data. Both of their countries count as home.
- Most home starts are British (446) and Italian (400), so the answer mostly describes those drivers.

### Q3: Which teams' cars break down the most?

**In plain words:** for each team, what share of its race starts ended because the car broke? We compare **2014-2021**
with **2022-2024**.

**Mechanical retirement** = a race start that ended early because **a part of the car failed** (engine, gearbox, brakes,
hydraulics, ...). Not counted: crashes, driver mistakes, and rule decisions (for example a disqualification).

**Where the data comes from:** everything comes from `load_clean()`.
- `model_base`: who started which race, for which team, in which year.
- `outcomes`: how each race ended (`statusId`). The team rule says `outcomes` is *only for looking at the data, never as
  model input*. Q3 only counts how races ended and builds no model, so this is exactly that allowed use.
- `tables["status"]`: the 139 status names, which we label by hand.
- `tables["constructors"]`: the team names.

**Tables and joins (multi-hop):**

```
model_base ──(raceId, driverId)──► outcomes ──statusId──► status        (how did the race end?)
     │
     └──constructorId──► constructors                                     (which team?)
```

`results → races` (for the year) is already done inside `load_clean()`. Not used: sprint results (a different, shorter
race), lap times, pit stops, standings.

In [ ]:
# how each status ended the race, written by hand from the status table (139 values)
ACCIDENT = ["Accident", "Collision", "Collision damage", "Damage", "Debris", "Fatal accident",
            "Puncture", "Tyre puncture", "Front wing", "Rear wing", "Broken wing", "Undertray"]
DRIVER = ["Spun off", "Stalled", "Illness", "Injured", "Injury", "Physical", "Driver unwell", "Eye injury"]
OTHER = ["Disqualified", "Excluded", "Underweight", "Retired", "Not classified", "Withdrew", "Did not qualify",
         "Did not prequalify", "107% Rule", "Not restarted", "Safety", "Safety concerns", "Out of fuel",
         "Refuelling", "Fuel rig", "Wheel nut"]

status_labels = tables["status"][["statusId", "status"]].copy()

# every status starts as "mechanical" (a part of the car failed), then the lists above overwrite it
status_labels["label"] = "mechanical"
status_labels.loc[status_labels["status"].isin(ACCIDENT), "label"] = "accident"
status_labels.loc[status_labels["status"].isin(DRIVER), "label"] = "driver"
status_labels.loc[status_labels["status"].isin(OTHER), "label"] = "other"

# finished the race: "Finished", or "+1 Lap", "+2 Laps", ... (classified, but some laps behind the winner)
finished = (status_labels["status"] == "Finished") | status_labels["status"].str.startswith("+")
status_labels.loc[finished, "label"] = "finished"

# check: every name in the lists really exists in the status table (catches typos)
names_in_lists = set(ACCIDENT + DRIVER + OTHER)
names_in_table = set(status_labels["status"])
assert names_in_lists <= names_in_table, "not in the status table: " + str(names_in_lists - names_in_table)

print("statuses labelled:", len(status_labels))
status_labels["label"].value_counts()

In [ ]:
# the full label table, one line per label (this is the documented mapping)
for label in ["mechanical", "accident", "driver", "other", "finished"]:
    names = status_labels[status_labels["label"] == label]["status"].tolist()
    print(label.upper(), "(" + str(len(names)) + "):", ", ".join(names))
    print()

#### Team renames

Some teams changed their name but stayed the same team (same factory, same staff). We put each name under one
**team history**, so a team's breakdowns are not split over two or three names.

In [ ]:
# team name in constructors.csv -> team history, written by hand (teams not listed keep their own name)
TEAM_HISTORY = {
    "Toro Rosso": "Toro Rosso / AlphaTauri / RB",
    "AlphaTauri": "Toro Rosso / AlphaTauri / RB",
    "RB F1 Team": "Toro Rosso / AlphaTauri / RB",
    "Force India": "Force India / Racing Point / Aston Martin",
    "Racing Point": "Force India / Racing Point / Aston Martin",
    "Aston Martin": "Force India / Racing Point / Aston Martin",
    "Lotus F1": "Lotus / Renault / Alpine",
    "Renault": "Lotus / Renault / Alpine",
    "Alpine F1 Team": "Lotus / Renault / Alpine",
    "Sauber": "Sauber / Alfa Romeo",
    "Alfa Romeo": "Sauber / Alfa Romeo",
    "Marussia": "Marussia / Manor",
    "Manor Marussia": "Marussia / Manor",
}
pd.DataFrame(list(TEAM_HISTORY.items()), columns=["name in the data", "team history"])

**Check: are these really the same team?** For every renamed team we print the seasons and the drivers. The old name
stops exactly when the new one starts, and the same drivers appear under both names (for example Gasly at Toro Rosso
and AlphaTauri, Pérez at Force India and Racing Point, Ocon at Renault and Alpine, Bottas and Zhou at Alfa Romeo and Sauber).

In [ ]:
# seasons and drivers of every renamed team, 2014-2024
check = base[base["year"] >= 2014]
check = check.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left")
check = check.merge(tables["drivers"][["driverId", "surname"]], on="driverId", how="left")

for name in TEAM_HISTORY:
    rows = check[check["name"] == name]
    seasons = sorted(set(rows["year"].tolist()))     # .tolist() gives plain numbers (2014 instead of np.int64(2014))
    drivers = sorted(set(rows["surname"].tolist()))
    print(TEAM_HISTORY[name], "|", name, "| seasons:", seasons, "| drivers:", ", ".join(drivers))

#### Build the Q3 table (2014-2024)

In [ ]:
# 1) who started which race for which team (2014-2024)
q3 = base[base["year"] >= 2014][["raceId", "driverId", "constructorId", "year"]]

# 2) joins: how the race ended (statusId) -> its status name and label -> the team name
q3 = q3.merge(outcomes[["raceId", "driverId", "statusId"]], on=["raceId", "driverId"], how="left", validate="one_to_one")
q3 = q3.merge(status_labels, on="statusId", how="left", validate="many_to_one")
q3 = q3.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left", validate="many_to_one")

# 3) safety checks
assert len(q3) == (base["year"] >= 2014).sum(), "a join changed the number of rows"
assert q3["label"].notna().all(), "a race end has no label"
assert q3["name"].notna().all(), "a team has no name"

# 4) team history (renames merged) and period
q3["team"] = q3["name"].replace(TEAM_HISTORY)
q3["period"] = "2014-21"
q3.loc[q3["year"] >= 2022, "period"] = "2022-24"

print("race starts:", len(q3), "| teams:", q3["team"].nunique())
print(q3.groupby("period")["raceId"].nunique())

#### Demo: how the Q3 joins work, step by step

Six real rows, one for each kind of race end, including two renamed teams (Aston Martin, AlphaTauri). Then two fake
rows show what would happen if a status or a team could not be found. These cells do not change `q3`.

In [ ]:
# DEMO (only to understand the joins): 6 real rows. Nothing here changes q3.
examples = [
    (963, 1),      # Hamilton, Malaysian GP 2016       -> Engine        -> mechanical
    (952, 3),      # Rosberg, Spanish GP 2016          -> Collision     -> accident
    (1076, 832),   # Sainz, Australian GP 2022         -> Spun off      -> driver
    (1062, 20),    # Vettel (Aston Martin), Hungary 2021 -> Disqualified -> other
    (1038, 842),   # Gasly (AlphaTauri), Italian GP 2020 -> Finished    -> finished (he won)
    (1137, 815),   # Pérez, Azerbaijan GP 2024         -> Retired       -> other (no reason given)
]
example_pairs = pd.DataFrame(examples, columns=["raceId", "driverId"])

# STEP 0: the tables alone (only the rows we need)
demo_base = base.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "constructorId", "year"]]
print("STEP 0a: BASE (model_base), our 6 rows")
display(demo_base)

demo_outcomes = outcomes.merge(example_pairs, on=["raceId", "driverId"])[["raceId", "driverId", "statusId"]]
print("STEP 0b: OUTCOMES, the same 6 rows (only statusId)")
display(demo_outcomes)

print("STEP 0c: STATUS_LABELS, only these statuses")
display(status_labels[status_labels["statusId"].isin(demo_outcomes["statusId"])])

print("STEP 0d: CONSTRUCTORS, only these teams")
display(tables["constructors"][tables["constructors"]["constructorId"].isin(demo_base["constructorId"])][["constructorId", "name"]])

# STEP 1: base + outcomes, matched on raceId AND driverId -> every row gets its statusId
step1 = demo_base.merge(outcomes[["raceId", "driverId", "statusId"]], on=["raceId", "driverId"], how="left")
print("STEP 1: base + outcomes (matched on raceId and driverId)")
display(step1)

# STEP 2: + status_labels, matched on statusId -> every row gets the status name and its label
step2 = step1.merge(status_labels, on="statusId", how="left")
print("STEP 2: + status_labels (matched on statusId)")
display(step2)

# STEP 3: + constructors, matched on constructorId -> team name, then team history and period
step3 = step2.merge(tables["constructors"][["constructorId", "name"]], on="constructorId", how="left")
step3["team"] = step3["name"].replace(TEAM_HISTORY)
step3["period"] = "2014-21"
step3.loc[step3["year"] >= 2022, "period"] = "2022-24"
print("STEP 3: + constructors (matched on constructorId), team history and period")
display(step3)

#### How did the races end?

Before the team rates, the big picture: how many starts in each period ended in each way.

In [ ]:
# count of starts per period and label
ended = pd.crosstab(q3["period"], q3["label"])
ended = ended[["finished", "mechanical", "accident", "driver", "other"]]
display(ended)

# the same as % of starts (normalize="index" makes every row add up to 100%)
ended_pct = pd.crosstab(q3["period"], q3["label"], normalize="index") * 100
ended_pct = ended_pct[["finished", "mechanical", "accident", "driver", "other"]]
ended_pct.round(1)

#### Mechanical retirement rate per team

**Rate** = mechanical retirements ÷ race starts × 100. Example: 46 breakdowns in 319 starts = 14.4%.

**Minimum 40 starts per team and period** (about one full season with 2 cars). A team with fewer starts raced only
part of a season, so a few breakdowns would give it an extreme rate. Teams below the minimum are shown but not ranked.

In [ ]:
MIN_TEAM_STARTS = 40   # about one full season with 2 cars

# True / False for every start: did it end with a car failure?
q3["mechanical"] = q3["label"] == "mechanical"

# per period and team: starts (count of rows) and breakdowns (sum of True = 1)
starts = q3.groupby(["period", "team"])["mechanical"].count()
breakdowns = q3.groupby(["period", "team"])["mechanical"].sum()

rates = pd.DataFrame({"starts": starts, "mechanical": breakdowns})
rates["rate %"] = rates["mechanical"] / rates["starts"] * 100
rates["enough starts"] = rates["starts"] >= MIN_TEAM_STARTS

print("left out (fewer than", MIN_TEAM_STARTS, "starts):")
display(rates[~rates["enough starts"]].round(1))
rates.round(1)

In [ ]:
# one bar chart per period, teams sorted by rate (the highest at the top)
for period in ["2014-21", "2022-24"]:
    one_period = rates.loc[period]                               # only this period (rows = teams)
    one_period = one_period[one_period["enough starts"]]         # only teams with enough starts
    one_period = one_period.sort_values("rate %")                # low to high, so barh draws the highest at the top

    plt.figure(figsize=(9, 5))
    plt.barh(one_period.index, one_period["rate %"], color="tab:red", label="mechanical retirements per 100 starts")
    plt.title("Mechanical retirement rate per team, " + period)
    plt.xlabel("% of race starts that ended with a car failure")
    plt.ylabel("team")
    plt.xlim(0, 16)                                              # same scale in both charts, so they can be compared
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
MIN_TEAM_STARTS = 30   # about one full season with 2 cars

# True / False for every start: did it end with a car failure?
q3["mechanical"] = q3["label"] == "mechanical"

# per period and team: starts (count of rows) and breakdowns (sum of True = 1)
starts = q3.groupby(["period", "team"])["mechanical"].count()
breakdowns = q3.groupby(["period", "team"])["mechanical"].sum()

rates = pd.DataFrame({"starts": starts, "mechanical": breakdowns})
rates["rate %"] = rates["mechanical"] / rates["starts"] * 100
rates["enough starts"] = rates["starts"] >= MIN_TEAM_STARTS

print("left out (fewer than", MIN_TEAM_STARTS, "starts):")
display(rates[~rates["enough starts"]].round(1))
rates.round(1)

In [ ]:
# one bar chart per period, teams sorted by rate (the highest at the top)
for period in ["2014-21", "2022-24"]:
    one_period = rates.loc[period]                               # only this period (rows = teams)
    one_period = one_period[one_period["enough starts"]]         # only teams with enough starts
    one_period = one_period.sort_values("rate %")                # low to high, so barh draws the highest at the top

    plt.figure(figsize=(9, 5))
    plt.barh(one_period.index, one_period["rate %"], color="tab:red", label="mechanical retirements per 100 starts")
    plt.title("Mechanical retirement rate per team, " + period)
    plt.xlabel("% of race starts that ended with a car failure")
    plt.ylabel("team")
    plt.xlim(0, 16)                                              # same scale in both charts, so they can be compared
    plt.legend(loc="lower right")
    plt.show()

#### Answer to Q3

**2014-2021, highest mechanical-retirement rate:**
1. Toro Rosso / AlphaTauri / RB: **14.4%** (46 of 319 starts)
2. Lotus / Renault / Alpine: 13.8% (44 of 318)
3. McLaren: 13.7% (43 of 315)

Lowest: Mercedes, **3.8%** (12 of 320).

**2022-2024, highest mechanical-retirement rate:**
1. Lotus / Renault / Alpine: **10.4%** (14 of 135)
2. Sauber / Alfa Romeo: 8.8% (12 of 136)
3. Toro Rosso / AlphaTauri / RB: 5.9% (8 of 136)

Lowest: Red Bull and McLaren, **3.7%** (5 of 136 each).

**What changed:**
- Cars broke down less in 2022-24. Mechanical retirements fell from 9.4% to 5.5% of all starts, while crashes stayed the
  same (6.9% → 6.7%). More cars finished (82.0% → 86.2%).
- Lotus / Renault / Alpine is near the top in both periods (2nd → 1st).
- McLaren dropped the most, from 13.7% (3rd) to 3.7% (lowest).

**Grain**
- Input: one row per driver per race start (`model_base`, 2014-2024: 4595 starts).
- Result: one row per team and period.
- All joins are checked: `outcomes` is one-to-one, `status` and `constructors` are many-to-one, and the row count stays the same.

**Missingness**
- Every start has a status, and all 139 status values have a label (checked with asserts).
- Teams with fewer than 40 starts in a period are shown but not ranked (only Caterham 2014: 7 of 33 = 21%).

**Pre-race cutoff**
- The status (how the race ended) is the *outcome* we measure. It comes from `outcomes` and is never used as a model input.

**Limitations**
- "Retired" without a reason counts as other, so a few breakdowns may be missed. It is a similar share in both periods
  (1.5% vs 1.2% of starts, inside "other"), so the comparison stays fair.
- Borderline labels are a choice: punctures and wing damage count as accidents (usually contact), a failed tyre counts as mechanical.
- Team renames are merged by hand (checked: same seasons in a row, same drivers).
- 2022-24 has only 3 seasons (about 135 starts per team), so one or two breakdowns move a team's rate by about 1 point.
- A breakdown is counted for the team, even when the failing part (for example the engine) comes from another manufacturer.